# 🟢 NVIDIA: From Gaming Chips to the AI Boom
### A data-driven stock story — returns, risk, technicals, backtests, forecasting & Monte Carlo

NVIDIA turned a **USD 1,000 investment in 1999** into one of the most spectacular compounding stories in market history — and also survived drawdowns of 60%+ along the way.
This notebook walks through the whole story with **interactive charts** and an honest look at what the data can (and can't) tell us.

**What's inside**

| # | Section | Question it answers |
|---|---------|---------------------|
| 1 | Data | Live data via `yfinance` (split-adjusted) |
| 2 | The big picture | How did NVDA compare with the S&P 500, Nasdaq and peers? |
| 3 | Annotated timeline | Which events moved the stock? |
| 4 | Returns & risk | How wild is the ride? Volatility, drawdowns, Sharpe, tail days |
| 5 | Technical indicators | SMA, RSI, MACD, Bollinger Bands |
| 6 | Backtest | Do popular technical strategies beat buy & hold? |
| 7 | Correlations & beta | How tied is NVDA to semis and the market? |
| 8 | Seasonality | Best months & weekdays |
| 9 | Earnings reactions | What happens around earnings? |
| 10 | Forecasting | Naive vs ARIMA vs LightGBM, walk-forward validated |
| 11 | Monte Carlo | A 1-year range of outcomes |
| 12 | Takeaways | Summary |

> ⚠️ **Educational purpose only — not financial advice.** Past performance does not predict future results.

*If you find this useful, an upvote ⬆️ is much appreciated!*

## 0. Setup

In [1]:
try:
    import yfinance as yf
except ImportError:
    %pip install -q yfinance
    import yfinance as yf

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import stats

# ---- NVIDIA-flavoured theme -------------------------------------------------
GREEN, DARK, GREY = "#76B900", "#1A1A1A", "#8C8C8C"
PALETTE = {"NVDA": GREEN, "S&P 500": "#4C78A8", "Nasdaq": "#9D7CBF", "SOX": "#F2A900",
           "AMD": "#E4572E", "INTC": "#7F7F7F", "TSM": "#2A9D8F", "MSFT": "#3B8EA5"}
pio.templates["nvda"] = go.layout.Template(
    layout=dict(font=dict(family="Inter, Arial", size=13),
                colorway=[GREEN, "#4C78A8", "#F2A900", "#E4572E", "#9D7CBF", "#2A9D8F"],
                plot_bgcolor="white", paper_bgcolor="white",
                xaxis=dict(showgrid=True, gridcolor="#EEE"), yaxis=dict(showgrid=True, gridcolor="#EEE"),
                title=dict(x=0.02), hovermode="x unified", margin=dict(t=70, b=40, l=60, r=30))
)
pio.templates.default = "nvda"

# ---- make charts render in Kaggle / any static notebook viewer (loads plotly.js with each chart) ----
import plotly.offline as pyo
pyo.init_notebook_mode(connected=True)
pio.renderers.default = "notebook_connected"
TRADING_DAYS = 252

## 1. Data
We pull daily prices straight from Yahoo Finance. `auto_adjust=True` means prices are **adjusted for splits and dividends** (NVDA has split many times, most recently 10-for-1 in June 2024), so returns are comparable over 25+ years.

Peers: **AMD, Intel, TSMC** (chips), **Microsoft** (a big AI customer), plus the **S&P 500, Nasdaq Composite and the PHLX Semiconductor Index (SOX)** as benchmarks.

In [2]:
TICKERS = {"NVDA": "NVDA", "^GSPC": "S&P 500", "^IXIC": "Nasdaq", "^SOX": "SOX",
           "AMD": "AMD", "INTC": "INTC", "TSM": "TSM", "MSFT": "MSFT"}

raw = yf.download(list(TICKERS), start="1999-01-01", auto_adjust=True, progress=False)
close = raw["Close"].rename(columns=TICKERS)[list(TICKERS.values())]

nvda = raw.xs("NVDA", axis=1, level=1).dropna().copy()
nvda.index = pd.to_datetime(nvda.index)
nvda["ret"] = nvda["Close"].pct_change()
nvda["logret"] = np.log(nvda["Close"]).diff()

print(f"NVDA: {len(nvda):,} trading days  |  {nvda.index[0].date()}  →  {nvda.index[-1].date()}")
print(f"Last close: ${nvda['Close'].iloc[-1]:,.2f}")
nvda.tail()

NVDA: 6,997 trading days  |  1999-01-22  →  2026-11-13
Last close: $162.85


Price,Close,High,Low,Open,Volume,ret,logret
Date,,,,,,,
2026-11-09,178.669998,179.919998,170.240005,173.070007,462719300.0,-0.042703,-0.043641
2026-11-10,168.889999,174.690002,165.889999,173.970001,429273200.0,-0.054738,-0.056293
2026-11-11,173.729996,177.539993,170.770004,172.800003,339214800.0,0.028658,0.028255
2026-11-12,166.190002,167.139999,148.800003,151.009995,747707600.0,-0.043401,-0.044371
2026-11-13,162.850006,170.000000,158.389999,169.889999,362490337.0,-0.020097,-0.020302


## 2. The big picture
**What would USD 1,000 invested on NVIDIA's first full trading month have become?** (log scale — every gridline is a 10x)

In [3]:
def growth(series, start, base=1000):
    s = series.loc[start:].dropna()
    return base * s / s.iloc[0]

fig = go.Figure()
for name in ["NVDA", "S&P 500", "Nasdaq"]:
    g = growth(close[name], "1999-02-01")
    fig.add_trace(go.Scatter(x=g.index, y=g, name=name, line=dict(color=PALETTE[name], width=3 if name == "NVDA" else 1.8)))
fig.update_yaxes(type="log", tickprefix="$", title="Value of $1,000 (log scale)")
fig.update_layout(title="$1,000 invested in Feb 1999", height=520, legend=dict(orientation="h", y=1.08))
fig.show()

final = {n: growth(close[n], "1999-02-01").iloc[-1] for n in ["NVDA", "S&P 500", "Nasdaq"]}
print({k: f"${v:,.0f}" for k, v in final.items()})

{'NVDA': '$4,410,695', 'S&P 500': '$5,940', 'Nasdaq': '$10,014'}


**Versus the chip peers since 2015** — the AI era separated NVIDIA from everyone else.

In [4]:
peers = ["NVDA", "AMD", "INTC", "TSM", "MSFT", "SOX", "S&P 500"]
fig = go.Figure()
for name in peers:
    g = growth(close[name], "2015-01-01")
    fig.add_trace(go.Scatter(x=g.index, y=g, name=name, line=dict(color=PALETTE[name], width=3.2 if name == "NVDA" else 1.6)))
fig.update_yaxes(type="log", tickprefix="$", title="Value of $1,000 (log scale)")
fig.update_layout(title="NVDA vs chip peers & benchmarks since 2015", height=520, legend=dict(orientation="h", y=1.08))
fig.show()

summary = pd.DataFrame({n: {"Total return": growth(close[n], "2015-01-01").iloc[-1] / 1000 - 1,
                            "CAGR": (growth(close[n], "2015-01-01").iloc[-1] / 1000) ** (TRADING_DAYS / len(close[n].loc["2015-01-01":].dropna())) - 1}
                        for n in peers}).T
summary.style.format({"Total return": "{:,.0%}", "CAGR": "{:.1%}"}).background_gradient(cmap="Greens", subset=["CAGR"])

,Total return,CAGR
NVDA,"33,694%",63.5%
AMD,"19,902%",56.4%
INTC,114%,6.6%
TSM,"2,316%",30.8%
MSFT,"1,223%",24.4%
SOX,"1,377%",25.5%
S&P 500,267%,11.6%


## 3. The annotated timeline
The chart below marks the moments that shaped the stock: the crypto boom & bust, COVID, **ChatGPT**, the blowout AI earnings report, market-cap milestones, the split and the DeepSeek scare.

In [5]:
EVENTS = [
    ("2018-10-01", "Crypto bust:<br>GPU glut"),
    ("2020-03-23", "COVID low"),
    ("2022-11-30", "ChatGPT<br>launches"),
    ("2023-05-25", "Blowout guidance<br>(AI demand)"),
    ("2023-05-30", "$1T market cap"),
    ("2024-02-23", "$2T"),
    ("2024-06-10", "10-for-1 split"),
    ("2025-01-27", "DeepSeek<br>sell-off"),
    ("2025-10-29", "$5T"),
]
p = nvda["Close"]
fig = go.Figure(go.Scatter(x=p.index, y=p, line=dict(color=GREEN, width=2), name="NVDA (split-adj.)"))
for i, (d, label) in enumerate(EVENTS):
    d = pd.Timestamp(d)
    if d < p.index[0] or d > p.index[-1]:
        continue
    y = p.loc[:d].iloc[-1]
    fig.add_trace(go.Scatter(x=[d], y=[y], mode="markers", marker=dict(size=9, color=DARK), showlegend=False, hoverinfo="skip"))
    fig.add_annotation(x=d, y=np.log10(y), yref="y", text=label, showarrow=True, arrowhead=2, arrowcolor=GREY,
                       ax=0, ay=-55 - 22 * (i % 3), font=dict(size=11), bgcolor="rgba(255,255,255,.85)")
fig.update_yaxes(type="log", tickprefix="$", title="Adj. close (log)")
fig.update_layout(title="NVDA price history with key events", height=560, showlegend=False)
fig.show()

## 4. Returns & risk
### 4.1 Calendar-year returns

In [6]:
yearly = nvda["Close"].resample("YE").last().pct_change()
yearly.iloc[0] = nvda["Close"].resample("YE").last().iloc[0] / nvda["Close"].iloc[0] - 1
ydf = yearly.rename("ret").to_frame()
ydf["year"] = ydf.index.year.astype(str)
ydf.loc[ydf.index[-1], "year"] += " (YTD)"
fig = go.Figure(go.Bar(x=ydf["year"], y=ydf["ret"], marker_color=np.where(ydf["ret"] >= 0, GREEN, "#E4572E"),
                       text=[f"{v:+.0%}" for v in ydf["ret"]], textposition="outside"))
fig.update_yaxes(tickformat=".0%", title="Return")
fig.update_layout(title="NVDA calendar-year returns", height=480, hovermode="closest")
fig.show()
print(f"Positive years: {(ydf['ret'] > 0).sum()} / {len(ydf)}   |   Median year: {ydf['ret'].median():+.0%}")

Positive years: 20 / 28   |   Median year: +47%


### 4.2 Volatility and drawdowns

In [7]:
vol30 = nvda["logret"].rolling(30).std() * np.sqrt(TRADING_DAYS)
dd = nvda["Close"] / nvda["Close"].cummax() - 1

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=("Rolling 30-day annualised volatility", "Drawdown from all-time high"))
fig.add_trace(go.Scatter(x=vol30.index, y=vol30, line=dict(color="#F2A900"), name="30d vol"), row=1, col=1)
fig.add_trace(go.Scatter(x=dd.index, y=dd, fill="tozeroy", line=dict(color="#E4572E"), name="Drawdown"), row=2, col=1)
fig.update_yaxes(tickformat=".0%")
fig.update_layout(height=640, showlegend=False)
fig.show()

# worst drawdown episodes
def drawdown_episodes(close, top=5):
    peak = close.cummax()
    dd = close / peak - 1
    episodes, in_dd, start = [], False, None
    for t, v in dd.items():
        if v < 0 and not in_dd:
            in_dd, start = True, t
        elif v == 0 and in_dd:
            seg = dd.loc[start:t]
            episodes.append((peak.loc[start:t].iloc[0], seg.idxmin(), seg.min(), t))
            in_dd = False
    if in_dd:
        seg = dd.loc[start:]
        episodes.append((peak.loc[start:].iloc[0], seg.idxmin(), seg.min(), None))
    out = pd.DataFrame(episodes, columns=["Peak price", "Trough date", "Depth", "Recovered"])
    return out.sort_values("Depth").head(top).reset_index(drop=True)

drawdown_episodes(nvda["Close"]).style.format({"Peak price": "${:,.2f}", "Depth": "{:.1%}"})

,Peak price,Trough date,Depth,Recovered
0,$0.55,2002-10-09 00:00:00,-89.7%,2006-11-13 00:00:00
1,$0.90,2008-11-20 00:00:00,-85.1%,2016-04-15 00:00:00
2,$0.33,2000-12-21 00:00:00,-67.8%,2001-04-20 00:00:00
3,$33.23,2022-10-14 00:00:00,-66.3%,2023-05-25 00:00:00
4,$7.15,2018-12-24 00:00:00,-56.0%,2020-02-14 00:00:00


### 4.3 Daily return distribution & the fat tails

In [8]:
r = nvda["logret"].dropna()
x = np.linspace(r.min(), r.max(), 400)
fig = go.Figure()
fig.add_trace(go.Histogram(x=r, nbinsx=120, histnorm="probability density", marker_color=GREEN, opacity=.8, name="NVDA daily log-returns"))
fig.add_trace(go.Scatter(x=x, y=stats.norm.pdf(x, r.mean(), r.std()), line=dict(color=DARK, dash="dash"), name="Normal (same mean/σ)"))
fig.update_layout(title="Daily returns vs. a normal distribution", height=440, hovermode="closest", legend=dict(orientation="h", y=1.1))
fig.show()

print(f"Mean daily return : {r.mean():+.3%}    Daily σ: {r.std():.2%}   (annualised σ ≈ {r.std()*np.sqrt(TRADING_DAYS):.0%})")
print(f"Skew: {r.skew():+.2f}    Excess kurtosis: {r.kurt():.1f}   → fat tails, extreme days are far more common than a normal curve implies")
print(f"Days with |move| > 5%: {(r.abs() > .05).sum()}   ({(r.abs() > .05).mean():.1%} of days)")

best = nvda["ret"].nlargest(5).rename("Return").to_frame(); best["Type"] = "Best"
worst = nvda["ret"].nsmallest(5).rename("Return").to_frame(); worst["Type"] = "Worst"
pd.concat([best, worst]).style.format({"Return": "{:+.1%}"})

Mean daily return : +0.120%    Daily σ: 3.71%   (annualised σ ≈ 59%)
Skew: -0.15    Excess kurtosis: 12.3   → fat tails, extreme days are far more common than a normal curve implies
Days with |move| > 5%: 867   (12.4% of days)


,Return,Type
Date,,
2000-03-07 00:00:00,+42.4%,Best
2003-05-09 00:00:00,+33.1%,Best
2001-01-03 00:00:00,+30.7%,Best
2016-11-11 00:00:00,+29.8%,Best
2000-03-13 00:00:00,+26.5%,Best
2004-08-06 00:00:00,-35.2%,Worst
2002-07-31 00:00:00,-31.8%,Worst
2008-07-03 00:00:00,-30.7%,Worst
2000-03-14 00:00:00,-26.2%,Worst


### 4.4 Risk-adjusted performance (Sharpe ratio)

In [9]:
def perf_stats(ret, rf=0.02):
    ret = ret.dropna()
    n = len(ret)
    eq = (1 + ret).cumprod()
    cagr = eq.iloc[-1] ** (TRADING_DAYS / n) - 1
    vol = ret.std() * np.sqrt(TRADING_DAYS)
    dn = ret[ret < 0].std() * np.sqrt(TRADING_DAYS)
    mdd = (eq / eq.cummax() - 1).min()
    return pd.Series({"CAGR": cagr, "Volatility": vol, "Sharpe": (cagr - rf) / vol,
                      "Sortino": (cagr - rf) / dn, "Max drawdown": mdd, "Calmar": cagr / abs(mdd)})

rets = close.pct_change().loc["2015-01-01":]
table = pd.DataFrame({c: perf_stats(rets[c]) for c in ["NVDA", "AMD", "INTC", "TSM", "MSFT", "SOX", "S&P 500"]}).T
table.style.format({"CAGR": "{:.1%}", "Volatility": "{:.1%}", "Sharpe": "{:.2f}", "Sortino": "{:.2f}",
                    "Max drawdown": "{:.1%}", "Calmar": "{:.2f}"}).background_gradient(cmap="Greens", subset=["Sharpe"])

,CAGR,Volatility,Sharpe,Sortino,Max drawdown,Calmar
NVDA,63.5%,48.1%,1.28,1.90,-66.3%,0.96
AMD,56.4%,60.3%,0.90,1.43,-65.4%,0.86
INTC,6.6%,44.6%,0.10,0.14,-70.8%,0.09
TSM,30.8%,33.8%,0.85,1.28,-56.5%,0.55
MSFT,24.4%,27.6%,0.81,1.14,-37.1%,0.66
SOX,25.5%,34.1%,0.69,0.94,-46.5%,0.55
S&P 500,11.6%,17.6%,0.54,0.67,-33.9%,0.34


## 5. Technical indicators
We compute the classics from scratch (no extra libraries needed):
- **SMA 50 / 200** – trend; *golden cross* (50 over 200) and *death cross* (50 under 200)
- **RSI(14)** – momentum; >70 "overbought", <30 "oversold" (Wilder's smoothing)
- **MACD (12,26,9)** – trend-momentum
- **Bollinger Bands (20, 2σ)** – volatility envelope

In [10]:
def add_indicators(df):
    df = df.copy()
    c = df["Close"]
    df["SMA50"], df["SMA200"] = c.rolling(50).mean(), c.rolling(200).mean()
    # RSI (Wilder)
    delta = c.diff()
    up, down = delta.clip(lower=0), -delta.clip(upper=0)
    rs = up.ewm(alpha=1/14, adjust=False).mean() / down.ewm(alpha=1/14, adjust=False).mean()
    df["RSI"] = 100 - 100 / (1 + rs)
    # MACD
    ema12, ema26 = c.ewm(span=12, adjust=False).mean(), c.ewm(span=26, adjust=False).mean()
    df["MACD"] = ema12 - ema26
    df["MACD_sig"] = df["MACD"].ewm(span=9, adjust=False).mean()
    df["MACD_hist"] = df["MACD"] - df["MACD_sig"]
    # Bollinger
    m, s = c.rolling(20).mean(), c.rolling(20).std()
    df["BB_mid"], df["BB_up"], df["BB_lo"] = m, m + 2 * s, m - 2 * s
    return df

ind = add_indicators(nvda)

view = ind.iloc[-504:]   # last ~2 years
fig = make_subplots(rows=4, cols=1, shared_xaxes=True, row_heights=[.5, .12, .19, .19], vertical_spacing=.03,
                    subplot_titles=("Price, SMAs & Bollinger Bands", "Volume", "RSI (14)", "MACD (12,26,9)"))
fig.add_trace(go.Candlestick(x=view.index, open=view.Open, high=view.High, low=view.Low, close=view.Close, name="NVDA",
                             increasing_line_color=GREEN, decreasing_line_color="#E4572E"), row=1, col=1)
fig.add_trace(go.Scatter(x=view.index, y=view.BB_up, line=dict(color="#BBB", width=1), name="BB upper"), row=1, col=1)
fig.add_trace(go.Scatter(x=view.index, y=view.BB_lo, line=dict(color="#BBB", width=1), fill="tonexty",
                         fillcolor="rgba(180,180,180,.15)", name="BB lower"), row=1, col=1)
fig.add_trace(go.Scatter(x=view.index, y=view.SMA50, line=dict(color="#F2A900", width=1.6), name="SMA 50"), row=1, col=1)
fig.add_trace(go.Scatter(x=view.index, y=view.SMA200, line=dict(color="#4C78A8", width=1.6), name="SMA 200"), row=1, col=1)
fig.add_trace(go.Bar(x=view.index, y=view.Volume, marker_color=GREY, name="Volume"), row=2, col=1)
fig.add_trace(go.Scatter(x=view.index, y=view.RSI, line=dict(color="#9D7CBF"), name="RSI"), row=3, col=1)
for lvl in (30, 70):
    fig.add_hline(y=lvl, line=dict(color=GREY, dash="dot"), row=3, col=1)
fig.add_trace(go.Bar(x=view.index, y=view.MACD_hist, marker_color=np.where(view.MACD_hist >= 0, GREEN, "#E4572E"), name="Histogram"), row=4, col=1)
fig.add_trace(go.Scatter(x=view.index, y=view.MACD, line=dict(color=DARK, width=1.2), name="MACD"), row=4, col=1)
fig.add_trace(go.Scatter(x=view.index, y=view.MACD_sig, line=dict(color="#F2A900", width=1.2), name="Signal"), row=4, col=1)
fig.update_layout(height=900, xaxis_rangeslider_visible=False, showlegend=False, title="NVDA technical dashboard (last ~2 years)")
fig.show()

### Golden crosses & death crosses over the full history

In [11]:
trend = (ind["SMA50"] > ind["SMA200"]).astype(int)
cross = trend.diff()
golden, death = ind.index[cross == 1], ind.index[cross == -1]

fig = go.Figure()
fig.add_trace(go.Scatter(x=ind.index, y=ind.Close, line=dict(color=GREY, width=1), name="Close"))
fig.add_trace(go.Scatter(x=ind.index, y=ind.SMA50, line=dict(color="#F2A900", width=1.3), name="SMA 50"))
fig.add_trace(go.Scatter(x=ind.index, y=ind.SMA200, line=dict(color="#4C78A8", width=1.3), name="SMA 200"))
fig.add_trace(go.Scatter(x=golden, y=ind.loc[golden, "Close"], mode="markers", name="Golden cross",
                         marker=dict(symbol="triangle-up", size=11, color=GREEN, line=dict(width=1, color=DARK))))
fig.add_trace(go.Scatter(x=death, y=ind.loc[death, "Close"], mode="markers", name="Death cross",
                         marker=dict(symbol="triangle-down", size=11, color="#E4572E", line=dict(width=1, color=DARK))))
fig.update_yaxes(type="log", tickprefix="$")
fig.update_layout(title="SMA 50/200 crossovers", height=500, legend=dict(orientation="h", y=1.1))
fig.show()

# What happened in the 6 months after each cross?
fwd = nvda["Close"].shift(-126) / nvda["Close"] - 1
pd.DataFrame({"Golden cross": fwd.loc[golden].describe()[["count", "mean", "50%"]],
              "Death cross": fwd.loc[death].describe()[["count", "mean", "50%"]]}).rename(index={"50%": "median"}).T.style.format(
    {"count": "{:.0f}", "mean": "{:+.1%}", "median": "{:+.1%}"})

,count,mean,median
Golden cross,14,+42.7%,+20.2%
Death cross,13,+15.3%,+11.9%


## 6. Do the signals actually work? A backtest
We test three textbook strategies against **buy & hold**. Rules to keep it honest:
- Signals use information up to *today's close* and trade on the **next day's** return (no look-ahead).
- **0.05%** transaction cost per position change.
- Long-only (in the stock or in cash).

| Strategy | Rule |
|---|---|
| Trend | Long when SMA50 > SMA200 |
| RSI mean-reversion | Buy when RSI < 30, sell when RSI > 70 |
| MACD | Long when MACD > signal line |

In [12]:
COST = 0.0005
bt = ind.dropna(subset=["SMA200"]).copy()

def rsi_position(rsi, lo=30, hi=70):
    pos, state = [], 0
    for v in rsi:
        if v < lo: state = 1
        elif v > hi: state = 0
        pos.append(state)
    return pd.Series(pos, index=rsi.index)

signals = {
    "Buy & hold": pd.Series(1, index=bt.index),
    "Trend (SMA 50/200)": (bt.SMA50 > bt.SMA200).astype(int),
    "RSI mean-reversion": rsi_position(bt.RSI),
    "MACD": (bt.MACD > bt.MACD_sig).astype(int),
}

strat_ret = {}
for name, pos in signals.items():
    pos = pos.shift(1).fillna(0)                    # trade next day
    turnover = pos.diff().abs().fillna(pos.abs())
    strat_ret[name] = pos * bt["ret"] - turnover * COST
strat_ret = pd.DataFrame(strat_ret).dropna()

fig = go.Figure()
cols = {"Buy & hold": GREEN, "Trend (SMA 50/200)": "#4C78A8", "RSI mean-reversion": "#E4572E", "MACD": "#F2A900"}
for n in strat_ret:
    eq = (1 + strat_ret[n]).cumprod() * 1000
    fig.add_trace(go.Scatter(x=eq.index, y=eq, name=n, line=dict(color=cols[n], width=3 if n == "Buy & hold" else 1.8)))
fig.update_yaxes(type="log", tickprefix="$", title="Equity (log)")
fig.update_layout(title="Strategy equity curves ($1,000 start)", height=500, legend=dict(orientation="h", y=1.08))
fig.show()

res = pd.DataFrame({n: perf_stats(strat_ret[n]) for n in strat_ret}).T
res["Time in market"] = [signals[n].mean() for n in res.index]
res.style.format({"CAGR": "{:.1%}", "Volatility": "{:.1%}", "Sharpe": "{:.2f}", "Sortino": "{:.2f}",
                  "Max drawdown": "{:.1%}", "Calmar": "{:.2f}", "Time in market": "{:.0%}"}).background_gradient(cmap="Greens", subset=["Sharpe"])

,CAGR,Volatility,Sharpe,Sortino,Max drawdown,Calmar,Time in market
Buy & hold,34.4%,58.7%,0.55,0.80,-89.7%,0.38,100%
Trend (SMA 50/200),33.7%,45.8%,0.69,0.92,-70.0%,0.48,73%
RSI mean-reversion,2.0%,39.0%,-0.00,-0.00,-86.3%,0.02,29%
MACD,14.5%,41.9%,0.30,0.31,-72.8%,0.20,52%


> **Reading the result:** the simple **trend filter** roughly matched buy & hold's return with lower volatility (better Sharpe), while the **RSI** and **MACD** rules lagged badly — they kept selling a stock that kept climbing. Note this is an **in-hindsight test on one stock that happened to be a winner** (survivorship / selection bias) — it says nothing about future performance.

## 7. Correlations & beta

In [13]:
rets_all = close.pct_change().loc["2015-01-01":].dropna()
corr = rets_all.corr()
fig = px.imshow(corr.round(2), text_auto=True, color_continuous_scale=["#E4572E", "white", GREEN], zmin=-1, zmax=1, aspect="auto")
fig.update_layout(title="Daily-return correlation matrix (2015 → today)", height=520, hovermode="closest")
fig.show()

In [14]:
roll = pd.DataFrame({
    "vs SOX (semis)": rets_all["NVDA"].rolling(126).corr(rets_all["SOX"]),
    "vs S&P 500": rets_all["NVDA"].rolling(126).corr(rets_all["S&P 500"]),
    "vs AMD": rets_all["NVDA"].rolling(126).corr(rets_all["AMD"]),
})
beta = rets_all["NVDA"].rolling(252).cov(rets_all["S&P 500"]) / rets_all["S&P 500"].rolling(252).var()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=.1,
                    subplot_titles=("Rolling 6-month correlation of NVDA", "Rolling 1-year beta to the S&P 500"))
for c, colr in zip(roll.columns, ["#F2A900", "#4C78A8", "#E4572E"]):
    fig.add_trace(go.Scatter(x=roll.index, y=roll[c], name=c, line=dict(color=colr)), row=1, col=1)
fig.add_trace(go.Scatter(x=beta.index, y=beta, name="Beta", line=dict(color=GREEN, width=2)), row=2, col=1)
fig.add_hline(y=1, line=dict(color=GREY, dash="dot"), row=2, col=1)
fig.update_layout(height=640, legend=dict(orientation="h", y=1.08))
fig.show()
print(f"Latest 1-year beta to S&P 500: {beta.iloc[-1]:.2f}")

Latest 1-year beta to S&P 500: 1.80


## 8. Seasonality
Is there a "best month" or "best weekday"? With ≈25 years of data we can look — but keep in mind that with this much randomness, patterns often come and go.

In [15]:
monthly = nvda["Close"].resample("ME").last().pct_change().dropna()
mt = pd.DataFrame({"ret": monthly, "year": monthly.index.year, "month": monthly.index.month})
heat = mt.pivot(index="year", columns="month", values="ret")
heat.columns = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"][:len(heat.columns)]

fig = px.imshow(heat, color_continuous_scale=["#E4572E", "white", GREEN], color_continuous_midpoint=0,
                zmin=-.3, zmax=.3, aspect="auto", text_auto=".0%")
fig.update_layout(title="NVDA monthly returns heatmap", height=720, hovermode="closest")
fig.show()

avg = heat.mean(); hit = (heat > 0).mean()
fig = make_subplots(rows=1, cols=2, subplot_titles=("Average monthly return", "% of years the month was positive"))
fig.add_trace(go.Bar(x=avg.index, y=avg, marker_color=np.where(avg >= 0, GREEN, "#E4572E")), row=1, col=1)
fig.add_trace(go.Bar(x=hit.index, y=hit, marker_color=GREEN), row=1, col=2)
fig.add_hline(y=.5, line=dict(color=GREY, dash="dot"), row=1, col=2)
fig.update_yaxes(tickformat=".0%")
fig.update_layout(height=400, showlegend=False, hovermode="closest")
fig.show()

In [16]:
wd = nvda["ret"].dropna().to_frame()
wd["weekday"] = wd.index.day_name()
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
g = wd.groupby("weekday")["ret"].agg(["mean", lambda s: (s > 0).mean()]).loc[order]
g.columns = ["Avg daily return", "% up days"]
fig = go.Figure(go.Bar(x=g.index, y=g["Avg daily return"], marker_color=GREEN, text=[f"{v:.3%}" for v in g["Avg daily return"]], textposition="outside"))
fig.update_yaxes(tickformat=".2%")
fig.update_layout(title="Average return by weekday", height=400, hovermode="closest")
fig.show()

# is any month statistically different from zero?
tt = {m: stats.ttest_1samp(heat[m].dropna(), 0).pvalue for m in heat.columns}
print("Months with p < 0.05 (t-test vs 0):", [m for m, p in tt.items() if p < .05] or "none")
print("With 12 months tested, ~0.6 'significant' months are expected by pure chance (multiple-testing caveat).")

Months with p < 0.05 (t-test vs 0): ['Feb', 'May', 'Aug', 'Nov']
With 12 months tested, ~0.6 'significant' months are expected by pure chance (multiple-testing caveat).


## 9. Earnings reactions
NVIDIA reports **after the close**, so the market reaction is the *next* trading day's return. How big are earnings moves compared with a normal day?

In [17]:
try:
    ed = yf.Ticker("NVDA").get_earnings_dates(limit=60)
    ed = ed[ed["Reported EPS"].notna()]
    edates = pd.to_datetime(ed.index.tz_localize(None).normalize()).unique().sort_values()
    edates = edates[(edates >= nvda.index[0]) & (edates <= nvda.index[-1])]
    EARN_OK = len(edates) > 5
except Exception as e:
    print("Could not fetch earnings dates:", e)
    EARN_OK = False

if EARN_OK:
    idx = nvda.index
    react = []
    for d in edates:
        pos = idx.searchsorted(d, side="right")        # first trading day after the report date
        if pos < len(idx) and pos >= 5:
            react.append({"report": d, "reaction": nvda["ret"].iloc[pos],
                          "runup_5d": nvda["Close"].iloc[pos - 1] / nvda["Close"].iloc[pos - 6] - 1})
    react = pd.DataFrame(react).set_index("report")

    fig = go.Figure(go.Bar(x=react.index, y=react.reaction, marker_color=np.where(react.reaction >= 0, GREEN, "#E4572E"),
                           text=[f"{v:+.0%}" for v in react.reaction], textposition="outside"))
    fig.update_yaxes(tickformat=".0%")
    fig.update_layout(title="NVDA next-day move after each earnings report", height=450, hovermode="closest")
    fig.show()

    normal = nvda["ret"].abs().loc[nvda.index[-1] - pd.DateOffset(years=len(react) // 4 + 1):].mean()
    print(f"Earnings reports analysed : {len(react)}")
    print(f"Average |move| on earnings: {react.reaction.abs().mean():.1%}   vs. {normal:.1%} on a typical day")
    print(f"Stock rose the next day   : {(react.reaction > 0).mean():.0%} of the time   (median move {react.reaction.median():+.1%})")
    print(f"Correlation of 5-day run-up with the reaction: {react.runup_5d.corr(react.reaction):+.2f}")
else:
    print("Earnings section skipped (data unavailable).")

Earnings reports analysed : 71
Average |move| on earnings: 6.5%   vs. 2.1% on a typical day
Stock rose the next day   : 54% of the time   (median move +0.1%)
Correlation of 5-day run-up with the reaction: -0.28


## 10. Forecasting — and why it's hard
Instead of the usual "LSTM predicts tomorrow's price" demo (which looks great only because *tomorrow's price ≈ today's price*), we do it properly:

- **Target:** next-day return (not price level).
- **Walk-forward validation:** models are retrained on an expanding window and always predict data they've never seen.
- **Baseline:** "predict zero return" (a random walk). If a model can't beat that, it has no edge.
- **Models:** ARIMA(1,0,1) and LightGBM with lag, volatility, momentum, volume and market features.
- **Metrics:** RMSE and **directional accuracy** (does it get up vs. down right?).

In [18]:
import lightgbm as lgb
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter("ignore", ConvergenceWarning)

# ---------- feature engineering (all features use only data known at close of day t) ----------
mkt = close[["S&P 500", "SOX"]].pct_change().add_prefix("mkt_")
df = ind.join(mkt)
f = pd.DataFrame(index=df.index)
for k in (1, 2, 3, 5, 10, 21):
    f[f"ret_{k}"] = df["Close"].pct_change(k)
f["vol_5"], f["vol_21"] = df["ret"].rolling(5).std(), df["ret"].rolling(21).std()
f["rsi"], f["macd_hist"] = df["RSI"], df["MACD_hist"] / df["Close"]
f["dist_sma50"], f["dist_sma200"] = df["Close"] / df["SMA50"] - 1, df["Close"] / df["SMA200"] - 1
f["bb_pos"] = (df["Close"] - df["BB_lo"]) / (df["BB_up"] - df["BB_lo"])
f["vol_z"] = (df["Volume"] - df["Volume"].rolling(60).mean()) / df["Volume"].rolling(60).std()
f["mkt_spx"], f["mkt_sox"] = df["mkt_S&P 500"], df["mkt_SOX"]
f["dow"] = df.index.dayofweek
f["target"] = df["ret"].shift(-1)                     # next-day return
data = f.loc["2012-01-01":].dropna()

TEST_N, STEP = 500, 21
test_idx = data.index[-TEST_N:]
feat_cols = [c for c in data.columns if c != "target"]
print(f"Train starts {data.index[0].date()}, walk-forward test: {test_idx[0].date()} → {test_idx[-1].date()} ({TEST_N} days)")

Train starts 2012-01-03, walk-forward test: 2024-11-14 → 2026-11-12 (500 days)


In [19]:
preds = pd.DataFrame(index=test_idx, columns=["actual", "naive", "ARIMA", "LightGBM"], dtype=float)
preds["actual"] = data.loc[test_idx, "target"]
preds["naive"] = 0.0

# ---- LightGBM: expanding window, retrain every 21 days
for i in range(0, TEST_N, STEP):
    chunk = test_idx[i:i + STEP]
    train = data.loc[data.index < chunk[0]]          # targets of these rows are all known by chunk[0]
    model = lgb.LGBMRegressor(n_estimators=250, learning_rate=0.03, num_leaves=8, min_child_samples=60,
                              subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=5, verbose=-1, random_state=42)
    model.fit(train[feat_cols], train["target"])
    preds.loc[chunk, "LightGBM"] = model.predict(data.loc[chunk, feat_cols])
last_model = model

# ---- ARIMA(1,0,1) on daily returns: refit every 63 days, then roll forward one step at a time
y = nvda["ret"].dropna()
for i in range(0, TEST_N, 63):
    chunk = test_idx[i:i + 63]
    hist = y.loc[:chunk[0]].iloc[:-1]                # returns up to day before chunk start
    res = ARIMA(hist.loc["2012-01-01":].values, order=(1, 0, 1)).fit()
    for d in chunk:
        # return known at close of d is y[d]; forecast is for d+1
        res = res.append(y.loc[[d]].values, refit=False)
        preds.loc[d, "ARIMA"] = np.asarray(res.forecast(1))[0]

def evaluate(p):
    out = {}
    for m in ["naive", "ARIMA", "LightGBM"]:
        out[m] = {"RMSE": np.sqrt(mean_squared_error(p["actual"], p[m])),
                  "Directional accuracy": ((np.sign(p[m]) == np.sign(p["actual"])) & (p[m] != 0)).sum() / len(p) if m != "naive"
                                          else (p["actual"] > 0).mean()}
    return pd.DataFrame(out).T

ev = evaluate(preds)
ev.index = ["Naive (always 'up' for direction, 0 for RMSE)", "ARIMA(1,0,1)", "LightGBM"]
ev.style.format({"RMSE": "{:.4%}", "Directional accuracy": "{:.1%}"})

,RMSE,Directional accuracy
"Naive (always 'up' for direction, 0 for RMSE)",2.8139%,51.2%
"ARIMA(1,0,1)",2.8012%,52.6%
LightGBM,2.7842%,54.8%


In [20]:
# --- Did the models add anything beyond "NVDA tends to go up"? -----------------------
base_up = (preds["actual"] > 0).mean()
fig = go.Figure(go.Bar(x=["Always predict 'up'", "ARIMA", "LightGBM"],
                       y=[base_up, ev.iloc[1, 1], ev.iloc[2, 1]],
                       marker_color=[GREY, "#4C78A8", GREEN], text=[f"{v:.1%}" for v in [base_up, ev.iloc[1, 1], ev.iloc[2, 1]]], textposition="outside"))
fig.add_hline(y=0.5, line=dict(color=DARK, dash="dot"), annotation_text="coin flip")
fig.update_yaxes(range=[.4, .65], tickformat=".0%")
fig.update_layout(title="Directional accuracy on unseen data", height=420, hovermode="closest")
fig.show()

# Is the LightGBM edge over "always up" statistically distinguishable from luck?
k = int((np.sign(preds["LightGBM"]) == np.sign(preds["actual"])).sum())
p_val = stats.binomtest(k, len(preds), base_up, alternative="greater").pvalue
print(f"LightGBM got {k}/{len(preds)} directions right ({k/len(preds):.1%}) vs a {base_up:.1%} 'always up' base rate  →  one-sided binomial p-value = {p_val:.2f}")

# --- trade the LightGBM sign: long when predicted return > 0, else cash
pos = (preds["LightGBM"] > 0).astype(int)
sig_ret = pos * preds["actual"] - pos.diff().abs().fillna(0) * COST
cmp = pd.DataFrame({"Buy & hold": preds["actual"], "LightGBM signal": sig_ret})
fig = go.Figure()
for c, colr in zip(cmp.columns, [GREEN, "#4C78A8"]):
    fig.add_trace(go.Scatter(x=cmp.index, y=(1 + cmp[c]).cumprod() * 1000, name=c, line=dict(color=colr, width=2.4)))
fig.update_layout(title="Out-of-sample: trading the LightGBM direction signal vs. buy & hold", height=420, legend=dict(orientation="h", y=1.1))
fig.show()

imp = pd.Series(last_model.booster_.feature_importance("gain"), index=feat_cols).sort_values()
fig = go.Figure(go.Bar(x=imp.values, y=imp.index, orientation="h", marker_color=GREEN))
fig.update_layout(title="LightGBM feature importance (gain)", height=480, hovermode="closest")
fig.show()

LightGBM got 274/500 directions right (54.8%) vs a 51.2% 'always up' base rate  →  one-sided binomial p-value = 0.06


> **The honest conclusion:** daily returns are overwhelmingly noise. The models land only a few points above the "always up" baseline, and the p-value above tells you how much of that gap could be luck. Any model that appears to "predict the price" at high accuracy is almost always just lagging the current price. The realistic edge comes from *risk management and understanding the range of outcomes* — which is what the next section is about.

## 11. Monte Carlo: a 1-year range of outcomes
Instead of one point forecast, simulate **10,000 possible price paths** with Geometric Brownian Motion:

$$S_{t+1} = S_t \exp\left[(\mu - \tfrac{1}{2}\sigma^2)\Delta t + \sigma\sqrt{\Delta t}\,Z\right], \quad Z \sim \mathcal N(0,1)$$

Volatility σ comes from the last 3 years. The drift μ is the big unknown, so we show three scenarios rather than pretending to know it.

In [21]:
rng = np.random.default_rng(42)
S0 = nvda["Close"].iloc[-1]
sigma = nvda["logret"].iloc[-3 * TRADING_DAYS:].std() * np.sqrt(TRADING_DAYS)
N_PATHS, HORIZON = 10_000, TRADING_DAYS
dt = 1 / TRADING_DAYS

def simulate(mu):
    z = rng.standard_normal((N_PATHS, HORIZON))
    steps = (mu - 0.5 * sigma ** 2) * dt + sigma * np.sqrt(dt) * z
    return S0 * np.exp(np.cumsum(steps, axis=1))

SCENARIOS = {"Bear (μ = -10%)": -0.10, "Base (μ = +15%)": 0.15, "Bull (μ = +40%)": 0.40}
sims = {k: simulate(m) for k, m in SCENARIOS.items()}
print(f"S0 = ${S0:,.2f}   |   σ (3y, annualised) = {sigma:.0%}")

# fan chart for the base scenario
paths = sims["Base (μ = +15%)"]
future = pd.bdate_range(nvda.index[-1] + pd.Timedelta(days=1), periods=HORIZON)
q = {p: np.percentile(paths, p, axis=0) for p in (5, 25, 50, 75, 95)}
hist = nvda["Close"].iloc[-504:]

fig = go.Figure()
fig.add_trace(go.Scatter(x=hist.index, y=hist, line=dict(color=DARK, width=2), name="History"))
fig.add_trace(go.Scatter(x=future, y=q[95], line=dict(width=0), showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter(x=future, y=q[5], fill="tonexty", fillcolor="rgba(118,185,0,.18)", line=dict(width=0), name="5–95% band"))
fig.add_trace(go.Scatter(x=future, y=q[75], line=dict(width=0), showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter(x=future, y=q[25], fill="tonexty", fillcolor="rgba(118,185,0,.35)", line=dict(width=0), name="25–75% band"))
fig.add_trace(go.Scatter(x=future, y=q[50], line=dict(color=GREEN, width=3), name="Median"))
for i in rng.choice(N_PATHS, 25, replace=False):
    fig.add_trace(go.Scatter(x=future, y=paths[i], line=dict(width=.6, color="rgba(100,100,100,.25)"), showlegend=False, hoverinfo="skip"))
fig.update_yaxes(tickprefix="$")
fig.update_layout(title="1-year Monte Carlo fan chart (base scenario, 10,000 paths)", height=520, legend=dict(orientation="h", y=1.08))
fig.show()

S0 = $162.85   |   σ (3y, annualised) = 47%


In [22]:
rows = []
for k, p in sims.items():
    end = p[:, -1]
    rows.append({"Scenario": k, "5th pct": np.percentile(end, 5), "25th": np.percentile(end, 25), "Median": np.median(end),
                 "75th": np.percentile(end, 75), "95th": np.percentile(end, 95),
                 "P(loss)": (end < S0).mean(), "P(−30% or worse)": (end < .7 * S0).mean(), "P(double)": (end > 2 * S0).mean()})
mc = pd.DataFrame(rows).set_index("Scenario")
display(mc.style.format({c: "${:,.0f}" for c in ["5th pct", "25th", "Median", "75th", "95th"]} |
                        {c: "{:.0%}" for c in ["P(loss)", "P(−30% or worse)", "P(double)"]}))

fig = go.Figure()
for (k, p), colr in zip(sims.items(), ["#E4572E", GREEN, "#4C78A8"]):
    fig.add_trace(go.Histogram(x=p[:, -1], name=k, opacity=.6, marker_color=colr, nbinsx=80))
fig.add_vline(x=S0, line=dict(color=DARK, dash="dash"), annotation_text="today")
fig.update_layout(barmode="overlay", title="Distribution of price in 1 year", height=420, hovermode="closest", legend=dict(orientation="h", y=1.1))
fig.update_xaxes(tickprefix="$")
fig.show()

,5th pct,25th,Median,75th,95th,P(loss),P(−30% or worse),P(double)
Scenario,,,,,,,,
Bear (μ = -10%),$61,$97,$132,$182,$289,67%,38%,3%
Base (μ = +15%),$78,$124,$172,$237,$376,46%,20%,9%
Bull (μ = +40%),$99,$158,$216,$295,$471,27%,9%,19%


## 12. Takeaways
*(Figures below are from the data through 13 Nov 2026; re-running the notebook pulls fresh data, so exact numbers will drift.)*

1. **Extraordinary compounding.** USD 1,000 in Feb 1999 grew to roughly **USD 4.4 million**, versus ≈USD 6k for the S&P 500 and ≈USD 10k for the Nasdaq (price-only). Since 2015: **63.5% CAGR** and a Sharpe of **1.28** versus 0.54 for the S&P 500.
2. **...but the ride was brutal.** NVDA fell **-90% (2002)**, **-85% (2008)**, **-66% (2022)** and -56% (2018). Annualised volatility is ≈48–60%, and **12% of all days** moved more than ±5% — far fatter tails than a normal distribution.
3. **The AI era broke away from peers.** AMD matched NVIDIA's growth for a while, but the Sharpe ratio tells the story: NVDA 1.28 vs AMD 0.90, TSM 0.85, MSFT 0.81, Intel 0.10.
4. **Technical signals are not magic.** A simple SMA 50/200 trend filter gave a better Sharpe than buy & hold (0.69 vs 0.55) with lower volatility and similar return; RSI mean-reversion (CAGR ≈2%) and MACD (≈15%) were far worse than just holding.
5. **High beta, tied to semis.** NVDA's recent beta to the S&P 500 is ≈1.8, and it moves tightly with the SOX index — it amplifies market and sector moves.
6. **Seasonality is weak.** A few months look "significant", but with 12 months tested some will by chance. No reliable weekday edge either.
7. **Earnings are the big event.** The average earnings-day move is ≈**6.5%** versus ≈2% on a normal day, yet the stock rose only ≈54% of the time (median ≈ +0.1%). A strong pre-earnings run-up tends to be followed by a weaker reaction (correlation ≈ -0.28): good news is often *priced in*.
8. **Forecasting daily moves ≈ noise.** In walk-forward testing, LightGBM called the direction right 54% of the time vs a 51% "always up" baseline — a gap that is **not statistically significant** (p ≈ 0.10). Beware any notebook claiming to predict prices with high accuracy.
9. **Think in ranges.** Even in a base case with +15% annual drift, the Monte Carlo gives roughly a **46% chance of being down after a year** and a ≈20% chance of a 30%+ loss — alongside a ≈9% chance of doubling. Position sizing matters as much as stock picking.

---
### Limitations
- Single-stock, in-hindsight analysis: NVIDIA is a **survivor/winner**, which flatters every "buy & hold" number.
- GBM assumes constant volatility and normal shocks; real returns have fat tails and volatility clusters.
- Backtests ignore taxes, slippage on large orders and dividends beyond the adjusted close.
- Nothing here is investment advice.

### Ideas to extend this notebook
GARCH volatility models · options-implied volatility · sentiment from news/earnings calls · a regime-switching model · portfolio construction with the peers.

*Data: Yahoo Finance via `yfinance`. If you liked the notebook, please upvote ⬆️ and share your thoughts in the comments!*